# BETO Fine-Tuning para Deteccion de Friccion EmocionalObjetivo: Fine-tunear BETO para predecir valencia y arousal en mensajes de incidentes.Arquitectura:- Capas 0-7 de BETO: congeladas- Capas 8-11 de BETO: entrenables- Head: Linear(768 -> 256 -> 2) para Valencia y ArousalRequisitos: GPU activada (Edit -> Notebook settings -> Hardware accelerator -> GPU)

In [ ]:
!nvidia-smiimport torchprint(f'PyTorch: {torch.__version__}')print(f'CUDA: {torch.cuda.is_available()}')print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
!pip install transformers scikit-learn pandas -qprint('Dependencias instaladas')

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osos.makedirs('/content/drive/MyDrive/beto_finetuned', exist_ok=True)print('Drive montado')

In [ ]:
import pandas as pdDATASET_PATH = '/content/drive/MyDrive/dataset_friccion_emocional.csv'if os.path.exists(DATASET_PATH):    df = pd.read_csv(DATASET_PATH)    print(f'Dataset cargado: {len(df)} mensajes')else:    print('Dataset no encontrado, usando ejemplo')    data = {'mensaje': ['otra vez la tabla sales_daily con datos inconsistentes', 'alguien me explica por que sigue sin cargar', 'si no carga antes del mediodia se me retrasan todos los informes', 'de mi lado no puedo hacer nada', 'excelente justo lo que necesitaba', 'hola vi algo raro en orders', 'esto no cuadra los numeros estan mal', 'ya no se que mas hacer solo me queda esperar', 'gracias por todo', 'todo esta caido no puedo trabajar'], 'valencia': [-0.70, -0.75, -0.60, -0.88, 0.55, -0.07, -0.65, -0.80, 0.50, -0.70], 'arousal': [0.55, 0.65, 0.30, -0.33, 0.15, 0.04, 0.55, -0.35, 0.20, 0.50], 'remitente': ['USR_01']*4 + ['USR_02']*3 + ['USR_03']*3}    df = pd.DataFrame(data)    print(f'Dataset ejemplo: {len(df)} mensajes')print(f'Columnas: {list(df.columns)}')

In [ ]:
import csv, os, timeimport numpy as npimport torchimport torch.nn as nnfrom torch.utils.data import Dataset, DataLoaderfrom transformers import AutoModel, AutoTokenizerfrom torch.optim import AdamWfrom torch.optim.lr_scheduler import LinearLRfrom sklearn.model_selection import GroupKFoldfrom sklearn.metrics import r2_score, mean_squared_errorDEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')BETO_MODEL = 'dccuchile/bert-base-spanish-wwm-cased'MAX_LEN = 128BATCH_SIZE = 16EPOCHS = 15LR = 2e-5DROPOUT = 0.1NUM_CAPAS_CONGELADAS = 8print(f'Device: {DEVICE}')print(f'Batch size: {BATCH_SIZE}')print(f'Epochs: {EPOCHS}')print(f'Learning rate: {LR}')print(f'Capas congeladas: {NUM_CAPAS_CONGELADAS}/12')class FriccionDataset(Dataset):    def __init__(self, textos, y_valencia, y_arousal, tokenizer, max_len=128):        self.textos = textos        self.y_v = y_valencia        self.y_a = y_arousal        self.tokenizer = tokenizer        self.max_len = max_len    def __len__(self):        return len(self.textos)    def __getitem__(self, idx):        text = self.textos[idx]        encoding = self.tokenizer(text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'y_valencia': torch.tensor(self.y_v[idx], dtype=torch.float), 'y_arousal': torch.tensor(self.y_a[idx], dtype=torch.float)}class BETORegressor(nn.Module):    def __init__(self, beto_model, dropout=0.1):        super().__init__()        self.beto = beto_model        self.dropout = nn.Dropout(dropout)        self.classifier = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 2))    def forward(self, input_ids, attention_mask):        outputs = self.beto(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)        cls_output = outputs.last_hidden_state[:, 0, :]        cls_output = self.dropout(cls_output)        logits = self.classifier(cls_output)        return logits[:, 0], logits[:, 1]def congelar_capas_bajas(model, num_capas_congeladas=8):    for param in model.embeddings.parameters():        param.requires_grad = False    for i, layer in enumerate(model.encoder.layer):        freeze = i < num_capas_congeladas        for param in layer.parameters():            param.requires_grad = not freeze    print(f'Capas 0-{num_capas_congeladas-1} congeladas, {num_capas_congeladas}-11 entrenables')print('Clases definidas')

In [ ]:
X = df['mensaje'].valuesy_v = df['valencia'].valuesy_a = df['arousal'].valuesgroups = df['remitente'].valuesprint(f'Dataset: {len(X)} mensajes, {len(np.unique(groups))} usuarios')print('Cargando BETO...')tokenizer = AutoTokenizer.from_pretrained(BETO_MODEL)beto_base = AutoModel.from_pretrained(BETO_MODEL)congelar_capas_bajas(beto_base, NUM_CAPAS_CONGELADAS)total_params = sum(p.numel() for p in beto_base.parameters())trainable_params = sum(p.numel() for p in beto_base.parameters() if p.requires_grad)print(f'Parametros totales: {total_params:,}')print(f'Parametros entrenables: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)')

In [ ]:
def entrenar_fold(model, train_loader, val_loader, device, epochs, lr, verbose=True):    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)    total_steps = len(train_loader) * epochs    scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)    criterion = nn.MSELoss()    best_val_loss = float('inf')    best_state = None    for epoch in range(epochs):        model.train()        train_loss = 0.0        for batch in train_loader:            input_ids = batch['input_ids'].to(device)            attention_mask = batch['attention_mask'].to(device)            y_v = batch['y_valencia'].to(device)            y_a = batch['y_arousal'].to(device)            optimizer.zero_grad()            pred_v, pred_a = model(input_ids, attention_mask)            loss = criterion(pred_v, y_v) + criterion(pred_a, y_a)            loss.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            optimizer.step()            scheduler.step()            train_loss += loss.item()        model.eval()        val_loss = 0.0        all_v, all_a, all_pv, all_pa = [], [], [], []        with torch.no_grad():            for batch in val_loader:                input_ids = batch['input_ids'].to(device)                attention_mask = batch['attention_mask'].to(device)                pred_v, pred_a = model(input_ids, attention_mask)                loss = criterion(pred_v, batch['y_valencia'].to(device)) + criterion(pred_a, batch['y_arousal'].to(device))                val_loss += loss.item()                all_v.extend(batch['y_valencia'].numpy())                all_a.extend(batch['y_arousal'].numpy())                all_pv.extend(pred_v.cpu().numpy())                all_pa.extend(pred_a.cpu().numpy())        r2_v = r2_score(all_v, all_pv)        r2_a = r2_score(all_a, all_pa)        avg_train = train_loss / len(train_loader)        avg_val = val_loss / len(val_loader)        if verbose:            print(f'Epoch {epoch+1}/{epochs} | train={avg_train:.4f} | val={avg_val:.4f} | R2v={r2_v:.3f} | R2a={r2_a:.3f}')        if avg_val < best_val_loss:            best_val_loss = avg_val            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}    if best_state:        model.load_state_dict(best_state)    return modelprint('Funcion de entrenamiento definida')

In [ ]:
print('VALIDACION CRUZADA (GroupKFold por remitente)')gkf = GroupKFold(n_splits=5)resultados = []for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_v, groups), 1):    print(f'Fold {fold}/5')    X_train, X_val = X[train_idx], X[val_idx]    yv_train, yv_val = y_v[train_idx], y_v[val_idx]    ya_train, ya_val = y_a[train_idx], y_a[val_idx]    train_dataset = FriccionDataset(X_train, yv_train, ya_train, tokenizer)    val_dataset = FriccionDataset(X_val, yv_val, ya_val, tokenizer)    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)    beto_copy = AutoModel.from_pretrained(BETO_MODEL)    congelar_capas_bajas(beto_copy, NUM_CAPAS_CONGELADAS)    model = BETORegressor(beto_copy).to(DEVICE)    start = time.time()    model = entrenar_fold(model, train_loader, val_loader, DEVICE, epochs=EPOCHS, lr=LR)    elapsed = time.time() - start    print(f'  Tiempo: {elapsed:.1f}s')    model.eval()    all_v, all_a, all_pv, all_pa = [], [], [], []    with torch.no_grad():        for batch in val_loader:            input_ids = batch['input_ids'].to(DEVICE)            attention_mask = batch['attention_mask'].to(DEVICE)            pred_v, pred_a = model(input_ids, attention_mask)            all_v.extend(batch['y_valencia'].numpy())            all_a.extend(batch['y_arousal'].numpy())            all_pv.extend(pred_v.cpu().numpy())            all_pa.extend(pred_a.cpu().numpy())    r2_v = r2_score(all_v, all_pv)    r2_a = r2_score(all_a, all_pa)    rmse_v = np.sqrt(mean_squared_error(all_v, all_pv))    rmse_a = np.sqrt(mean_squared_error(all_a, all_pa))    print(f'  R2_v={r2_v:.3f} | R2_a={r2_a:.3f} | RMSE_v={rmse_v:.3f} | RMSE_a={rmse_a:.3f}')    resultados.append({'fold': fold, 'r2_v': r2_v, 'r2_a': r2_a, 'rmse_v': rmse_v, 'rmse_a': rmse_a})print('RESUMEN VALIDACION HONESTA')r2_v_mean = np.mean([r['r2_v'] for r in resultados])r2_v_std = np.std([r['r2_v'] for r in resultados])r2_a_mean = np.mean([r['r2_a'] for r in resultados])r2_a_std = np.std([r['r2_a'] for r in resultados])print(f'Valencia  R2: {r2_v_mean:.3f} +/- {r2_v_std:.3f}')print(f'Arousal   R2: {r2_a_mean:.3f} +/- {r2_a_std:.3f}')print(f'Congelado + Ridge:  R2=0.943/0.907')print(f'Fine-tuned + Head:  R2={r2_v_mean:.3f}/{r2_a_mean:.3f}')

In [ ]:
print('ENTRENAMIENTO FINAL (100% datos)')full_dataset = FriccionDataset(X, y_v, y_a, tokenizer)full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)beto_final = AutoModel.from_pretrained(BETO_MODEL)congelar_capas_bajas(beto_final, NUM_CAPAS_CONGELADAS)model_final = BETORegressor(beto_final).to(DEVICE)start = time.time()model_final = entrenar_fold(model_final, full_loader, full_loader, DEVICE, epochs=20, lr=LR)elapsed = time.time() - startprint(f'Tiempo total: {elapsed:.1f}s ({elapsed/60:.1f} min)')print('GUARDANDO MODELO')save_path = '/content/drive/MyDrive/beto_finetuned'os.makedirs(save_path, exist_ok=True)torch.save(model_final.state_dict(), f'{save_path}/beto_finetuned.pt')tokenizer.save_pretrained(f'{save_path}/tokenizer')print(f'Guardado en Google Drive: {save_path}')local_path = '/content/beto_finetuned'os.makedirs(local_path, exist_ok=True)torch.save(model_final.state_dict(), f'{local_path}/beto_finetuned.pt')tokenizer.save_pretrained(f'{local_path}/tokenizer')print(f'Guardado en local: {local_path}')

In [ ]:
import shutilfrom google.colab import fileszip_path = '/content/beto_finetuned'shutil.make_archive(zip_path, 'zip', zip_path)print('Descargando beto_finetuned.zip...')files.download(f'{zip_path}.zip')print('Descarga iniciada. Guarda el archivo en tu PC.')

In [ ]:
print('PRUEBA DEL MODELO FINE-TUNED')model_prueba = BETORegressor(AutoModel.from_pretrained(BETO_MODEL)).to(DEVICE)model_prueba.load_state_dict(torch.load('/content/beto_finetuned/beto_finetuned.pt'))model_prueba.eval()pruebas = [    'otra vez la tabla de ventas con datos malos',    'estoy harto, todo caido, no puedo trabajar',    'excelente, justo lo que necesitaba',    'hola, buenos dias',    'si no carga antes del mediodia se me retrasan todos los informes',    'ya no se que hacer, solo me queda esperar',    'gracias por la ayuda',    'estas retrasando mi trabajo y no puedo avanzar',]for text in pruebas:    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN)    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}    with torch.no_grad():        v, a = model_prueba(inputs['input_ids'], inputs['attention_mask'])    v = v.cpu().item()    a = a.cpu().item()    sev = (v**2 + a**2)**0.5 / (2**0.5)    emo = 'frustracion' if v < -0.5 and a > 0.4 else 'preocupacion' if v < -0.5 and a > 0.2 else 'impotencia' if v < -0.5 and a < 0.2 else 'positivo' if v > 0.3 else 'neutral'    print(f'{text[:45]}... -> {emo:12s} v={v:+.2f} a={a:+.2f} sev={sev:.2f}')print('Fine-tuning completado!')print('Guarda el modelo en Google Drive y descarga beto_finetuned.zip')

## Resumen1. Verificar GPU2. Instalar dependencias3. Montar Drive4. Cargar dataset5. Definir clases6. Cargar datos y BETO7. Entrenar y validar8. Guardar modelo9. Descargar ZIP10. Probar modelo## Resultados esperados| Metrica | Congelado | Fine-tuned ||---------|-----------|------------|| Valencia R2 | 0.943 | 0.95-0.98 || Arousal R2 | 0.907 | 0.92-0.95 |## Notas- Si se desconecta la sesion: Guarda en Drive- Si te asignan GPU K80: Reinicia sesion hasta que te den T4- Dataset pequeno (< 100 mensajes): Puede overfitear. Usa dropout mas alto (0.3)